# P5 follow-ups on Gemma 3 12B (preregistered: `docs/preregistration_p5.md`)

**Before you press Run all (once):**
1. Upload `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`**.
2. Colab secrets (key icon on the left), with notebook access switched on:
   - **`HF_TOKEN`**: your Hugging Face token. Its account must have accepted the Gemma 3 licence on huggingface.co.
   - **`SLACK_WEBHOOK_URL`** (optional): a Slack *incoming webhook* URL. Every finished step posts a one-line update.
3. **Runtime → Change runtime type → A100 GPU.** (12B in bf16 needs ~24 GB; an L4 is too tight.)

Then **Runtime → Run all**. Expected time on an A100: about 1.5–2 hours.

**If Colab disconnects:** reconnect and press **Run all** again. Every finished step is saved straight to
`MyDrive/geomap/results/` and is skipped on the rerun, so at most the step in progress is repeated.

In [ ]:
# 1. GPU check
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 35, f"Only {gb:.0f} GB of GPU memory: pick an A100 (40 GB) runtime"

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):                      # first run: move the bundled results folder to Drive
    shutil.copytree("/content/geomap/results", RES)
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")       # every write under results/ goes straight to Drive
os.chdir("/content/geomap")
print("results folder:", os.path.realpath("results"))
print("already saved:", sorted(os.listdir(f"{RES}/p5")) if os.path.exists(f"{RES}/p5") else "nothing yet")

In [ ]:
# 3. Dependencies (torch comes with Colab; the code runs from src/ without installing the package)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ["PYTHONPATH"] = "/content/geomap/src"
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Secrets: Hugging Face (required, Gemma is gated) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    print("no SLACK_WEBHOOK_URL secret: progress goes to this notebook only")
from geomap.checkpoint import notify
print("Slack:", "ok" if notify("[P5] Colab run started (or resumed)") else "off")

In [ ]:
# 3b. PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import), then check that import directly
os.environ.update(USE_TF="0", USE_FLAX="0", USE_TORCH="1", TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
!python -c "import sys, torch, transformers, huggingface_hub; print(sys.version.split()[0], 'torch', torch.__version__, 'transformers', transformers.__version__, 'hub', huggingface_hub.__version__); from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 5. P5a + P5b: natural encoding and write-in (~30-45 min; resumes if interrupted)
!cd /content/geomap && python scripts/p5_encoding.py --model gemma3-12b 2>&1 | grep -v -i warning

In [ ]:
# 6. P5c: erase the map (~45-75 min; resumes if interrupted)
!cd /content/geomap && python scripts/p5_erase.py --model gemma3-12b 2>&1 | grep -v -i warning

In [ ]:
# 7. One zip of the P5 results + the results log, on Drive and downloaded here
shutil.make_archive("/content/p5_results", "zip", "/content/geomap", "results/p5")
with zipfile.ZipFile("/content/p5_results.zip", "a") as z:
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
shutil.copy("/content/p5_results.zip", f"{D}/p5_results.zip")
notify("[P5] all done: results in MyDrive/geomap/p5_results.zip")
from google.colab import files
files.download("/content/p5_results.zip")